

### Ziel dieser Datei
Benötigte Rohdaten für gesamte Schweiz runterladen. Dann filtern auf die relevanten Attribute (Wiese, Wälder, Schutzzonen usw). Dann auch sicherstellen, dass alle Datensätze in LV95 sind.

### Datenquellen
OSM-Daten über OverpassTurbo die jeweiligen Daten holen. Korrekte Abfragefilter machen, damit die richtigen Daten heruntergeladen werden!
  * Wiesen/Wälder (z.B. `landuse=meadow`, `landuse=forest`, usw.)
  * Infrastruktur (z.B. `amenity=farm`, `landuse=farmyard` für Bauernhöfe; `highway=bus_stop`, `railway=station` für ÖV)
  * Wenn möglich Geländeneigung, sonst mindestens natürliche Gefahrenzonen (`natural=cliff` für Felswände, `natural=scree` für Geröll) um dann diese Gebiete später auszuschliessen
Schutzgebiete Vektordatensatz als GeoJSON/Geopackage von geo.admin.ch herunterladen.

### Grober Codeaufbau
1. GeoJSON-Dateien einlesen
2. Koordinatensystem anpassen in LV95 (für berechnungen / verschnitte später)
3. Attributtabellen bereinigen: Unnötige Spalten löschen, damit die Dateien performant laufen.

### Export & Übernahme für die Nächste Datei 2
* Bereinigte Daten einzeln als GeoJSON abspeichern!
* Saubere Benennung nach Thematik, dass für Import in Datei 2 alles klar ist. z.B. `bearbeitet_flächen.geojson` / `bearbeitet_schutzgebiete.geojson`.

In [ ]:
import osmnx as ox
import geopandas as gpd
import warnings

# Warnungen unterdrücken für eine saubere Ausgabe
warnings.filterwarnings('ignore')

# 1. OSM-Daten beziehen
# HINWEIS: Für den finalen Lauf "Switzerland" verwenden. 
# Für Tests während des Hackathons besser einen kleineren Bereich (z.B. "Canton of Basel-Landschaft, Switzerland") wählen.
place_name = "Kanton Basel-Landschaft, Switzerland"

print("Lade Wälder und Wiesen...")
tags_nature = {'landuse': ['meadow', 'forest']}
gdf_nature = ox.features_from_place(place_name, tags_nature)

print("Lade Infrastruktur (Bauernhöfe, ÖV)...")
tags_infra = {
    'amenity': ['farm'],
    'landuse': ['farmyard'],
    'highway': ['bus_stop'],
    'railway': ['station']
}
gdf_infra = ox.features_from_place(place_name, tags_infra)


print("Lade Hydrantenstandorte herunter...")
tags_hydrants = {'emergency': 'fire_hydrant'}
gdf_hydrants = ox.features_from_place(place_name, tags_hydrants)





print("Lade Gefahrenzonen...")
tags_hazards = {'natural': ['cliff', 'scree']}
gdf_hazards = ox.features_from_place(place_name, tags_hazards)

# 2. Schutzgebiete von geo.admin.ch einlesen
# Hier den korrekten Pfad zur lokal heruntergeladenen Datei anpassen!
print("Lade Schutzgebiete...")
# gdf_schutz = gpd.read_file('pfad/zu/schutzgebiete.geojson') 

# 3. Koordinatensystem in LV95 (EPSG:2056) transformieren
print("Transformiere in LV95...")
gdf_nature_lv95 = gdf_nature.to_crs(epsg=2056)
gdf_infra_lv95 = gdf_infra.to_crs(epsg=2056)
gdf_hazards_lv95 = gdf_hazards.to_crs(epsg=2056)
gdf_hydrants_lv95 = gdf_hydrants.to_crs(epsg=2056)
# gdf_schutz_lv95 = gdf_schutz.to_crs(epsg=2056)

# 4. Attributtabellen bereinigen (Unnötige Spalten löschen für mehr Performance)
def clean_attributes(gdf, keep_columns):
    # Die Spalte 'geometry' muss zwingend erhalten bleiben
    existing_cols = [col for col in keep_columns if col in gdf.columns] + ['geometry']
    return gdf[existing_cols]

print("Bereinige Attributtabellen...")
gdf_nature_clean = clean_attributes(gdf_nature_lv95, ['landuse'])
gdf_infra_clean = clean_attributes(gdf_infra_lv95, ['amenity', 'landuse', 'highway', 'railway', 'name'])
gdf_hazards_clean = clean_attributes(gdf_hazards_lv95, ['natural'])
gdf_hydrants_clean = clean_attributes(gdf_hydrants_lv95, ['emergency'])

# 5. Export als bereinigte GeoJSON-Dateien für Datei 2
print("Exportiere GeoJSON-Dateien...")
gdf_nature_clean.to_file("data_BL/bearbeitet_flaechen.geojson", driver="GeoJSON")
gdf_infra_clean.to_file("data_BL/bearbeitet_infrastruktur.geojson", driver="GeoJSON")
gdf_hazards_clean.to_file("data_BL/bearbeitet_gefahrenzonen.geojson", driver="GeoJSON")
gdf_hydrants_clean.to_file("data_BL/bearbeitet_hydranten.geojson", driver="GeoJSON")
# gdf_schutz_lv95.to_file("bearbeitet_schutzgebiete.geojson", driver="GeoJSON")

print("Datenbezug und Export erfolgreich abgeschlossen!")

Lade Wälder und Wiesen...
Lade Infrastruktur (Bauernhöfe, ÖV)...
Lade Gefahrenzonen...
Lade Schutzgebiete...
Transformiere in LV95...
Bereinige Attributtabellen...
Exportiere GeoJSON-Dateien...
Datenbezug und Export erfolgreich abgeschlossen!
